# Weights configuration for Jaccard Index generation

In this notebook, I will be doing experimentation, training of models, and evaluation of different weight configurations for generating the Jaccard Index.


In [1]:
import psycopg
import optuna
import igraph as ig
import dotenv
import os
dotenv.load_dotenv()

/home/markb/root-py-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
conn = psycopg.connect(f"dbname={os.getenv('DB_NAME')} user={os.getenv('DB_USER')} host=127.0.0.1 port={os.getenv('DB_PORT')} password={os.getenv('DB_PASSWORD')}")
conn.read_only = True
cur = conn.cursor()


In [ ]:
START_DATE = '2026-08-27'
END_DATE = '2026-09-27'
FEATURE_NAMES = [
    'method', 'host', 'query_part', 'path_part', 'body_part', 'header', 'user_agent',
    'request_length', 'status_code', 'server_protocol', 'server_name', 'remote_addr',
    'accept_encoding', 'content_type', 'scheme'
]

In [4]:
cur.execute('SELECT * FROM part_types;')
print([desc[0] for desc in cur.description])
part_types = cur.fetchall()
print(part_types)

['id', 'name', 'description']
[(UUID('07901ba7-64d3-4603-9f5c-38a761ea9ed6'), 'method', 'HTTP Method (GET, POST, etc.)'), (UUID('e1b77c40-928f-4a87-9130-5ce6aeb91b62'), 'host', 'The host of the request'), (UUID('d3b24ec8-85e9-414a-9639-0d4b489388d9'), 'query_part', 'Part of the query part of the URL'), (UUID('6ff317f1-d8ec-48f0-8c6a-5e146c799185'), 'path_part', 'Part of the endpoint path of the URL'), (UUID('a515a5f3-4354-4186-b264-f3b616ab8197'), 'body_part', 'Part of the body of the request'), (UUID('eaec0218-0f1d-487b-9b63-4543a6422e36'), 'header', 'HTTP Header (User-Agent, Content-Type, etc.)'), (UUID('2165835b-b1b1-4da3-bbf6-683d0e82856d'), 'user_agent', 'The user agent of the request'), (UUID('756a9262-c513-4cc4-9f3f-3730f4055d35'), 'request_length', 'Length of the HTTP request'), (UUID('d8894b05-3053-4c45-9f58-03ebb3ee2f15'), 'status_code', 'HTTP Status Code'), (UUID('8ac61f8d-81b3-4e29-bb9e-13bfb22974ed'), 'server_protocol', 'The protocol used by the server (HTTP/1.1, HTTP/2, e

In [5]:
query = """
WITH selected_requests AS (
    SELECT id
    FROM requests
    WHERE created_at BETWEEN %s AND %s
),
weights AS (
    SELECT x.name, x.weight
    FROM jsonb_to_recordset(%s::jsonb) AS x(name text, weight double precision)
), shared_parts AS (
    SELECT rp1.request_id AS request_1,
        rp2.request_id AS request_2,
        SUM(w.weight) AS shared_parts_count
    FROM request_to_parts rp1
    JOIN request_to_parts rp2 ON rp1.part_id = rp2.part_id
    JOIN request_parts p ON rp1.part_id = p.id
    JOIN part_types pt ON pt.id = p.part_type
    JOIN weights w ON w.name = pt.name
    JOIN selected_requests sr1 ON rp1.request_id = sr1.id
    JOIN selected_requests sr2 ON rp2.request_id = sr2.id
    WHERE p.excluded = FALSE
      AND rp1.request_id < rp2.request_id
    GROUP BY rp1.request_id, rp2.request_id
),
non_excluded_parts AS (
    SELECT r.id AS request_id, SUM(w.weight) AS non_excluded_count
    FROM selected_requests r
    JOIN request_to_parts rtp ON r.id = rtp.request_id
    JOIN request_parts rp ON rtp.part_id = rp.id
    JOIN part_types pt ON pt.id = rp.part_type
    JOIN weights w ON w.name = pt.name
    WHERE rp.excluded = FALSE 
    GROUP BY r.id
),
full_graph AS (
    SELECT r1.ext_id AS request_1,
        r2.ext_id AS request_2,
        sp.shared_parts_count::float / (
            nep1.non_excluded_count + nep2.non_excluded_count - sp.shared_parts_count
        )::float AS jaccard_index
    FROM shared_parts sp
    JOIN requests r1 ON sp.request_1 = r1.id
    JOIN requests r2 ON sp.request_2 = r2.id
    JOIN non_excluded_parts nep1 ON sp.request_1 = nep1.request_id
    JOIN non_excluded_parts nep2 ON sp.request_2 = nep2.request_id
)
SELECT * FROM full_graph
WHERE jaccard_index > (SELECT value::float FROM loggraph_config WHERE key = 'min_jaccard_index');
"""
testWeights = [{"name": name, "weight": 1.0} for name in FEATURE_NAMES]
data = cur.execute(query,
                   (START_DATE, END_DATE,
                    psycopg.types.json.Jsonb(testWeights)
                    )).fetchall()
print(len(data))

1773820


In [6]:
TARGET_CLUSTERS = 50
MIN_WEIGHT = 0.0
MAX_WEIGHT = 5.0
WEIGHT_STEP = 0.1

def target_function(num_clusters: int):
    return (num_clusters - TARGET_CLUSTERS) ** 2

def evaluate_weights(weights: list[dict[str, float]]) -> tuple[float, int]:
    data = cur.execute(query, (
        START_DATE, END_DATE,
        psycopg.types.json.Jsonb(
            weights
        )
    )).fetchall()
    graph = ig.Graph.TupleList(data, directed=False, edge_attrs=['weight'])
    clusters = graph.community_leiden(weights='weight')
    cluster_count = len(clusters)
    score = target_function(cluster_count)
    return score, cluster_count

def objective(trial: optuna.Trial) -> float:
    trial_weights = [
        {
            "name": name,
            "weight": trial.suggest_float(name, MIN_WEIGHT, MAX_WEIGHT, step=WEIGHT_STEP)
        }
        for name in FEATURE_NAMES
    ]
    score, cluster_count = evaluate_weights(trial_weights)
    trial.set_user_attr('cluster_count', cluster_count)
    return score

evaluate_weights(testWeights)

(712943401, 26751)

In [7]:
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_weights = [{"name": name, "weight": float(value)} for name, value in study.best_params.items()]
best_score, best_cluster_count = evaluate_weights(best_weights)

print(f'Best score: {best_score}')
print(f'Best cluster count: {best_cluster_count}')
print('Best weights:')
print(best_weights)

[I 2026-11-23 04:17:16,093] A new study created in memory with name: no-name-331bfe3c-d2b2-4305-b524-f47eaec68e14
Best trial: 0. Best value: 7.14546e+08:   2%|▏         | 1/50 [06:11<5:03:39, 371.82s/it]

[I 2026-11-23 04:23:27,912] Trial 0 finished with value: 714546361.0 and parameters: {'method': 1.9000000000000001, 'host': 4.800000000000001, 'query_part': 3.7, 'path_part': 3.0, 'body_part': 0.7000000000000001, 'header': 0.7000000000000001, 'user_agent': 0.2, 'request_length': 4.4, 'status_code': 3.0, 'server_protocol': 3.6, 'server_name': 0.1, 'remote_addr': 4.9, 'accept_encoding': 4.2, 'content_type': 1.0, 'scheme': 0.9}. Best is trial 0 with value: 714546361.0.


Best trial: 1. Best value: 7.10116e+08:   4%|▍         | 2/50 [11:53<4:43:12, 354.01s/it]

[I 2026-11-23 04:29:09,465] Trial 1 finished with value: 710115904.0 and parameters: {'method': 0.9, 'host': 1.5, 'query_part': 2.6, 'path_part': 2.2, 'body_part': 1.4000000000000001, 'header': 3.1, 'user_agent': 0.7000000000000001, 'request_length': 1.4000000000000001, 'status_code': 1.8, 'server_protocol': 2.3000000000000003, 'server_name': 4.0, 'remote_addr': 1.0, 'accept_encoding': 2.6, 'content_type': 3.0, 'scheme': 0.2}. Best is trial 1 with value: 710115904.0.


Best trial: 2. Best value: 7.05168e+08:   6%|▌         | 3/50 [17:37<4:33:49, 349.55s/it]

[I 2026-11-23 04:34:53,711] Trial 2 finished with value: 705168025.0 and parameters: {'method': 3.0, 'host': 0.8, 'query_part': 0.30000000000000004, 'path_part': 4.800000000000001, 'body_part': 4.9, 'header': 4.1000000000000005, 'user_agent': 1.5, 'request_length': 0.4, 'status_code': 3.4000000000000004, 'server_protocol': 2.2, 'server_name': 0.6000000000000001, 'remote_addr': 2.5, 'accept_encoding': 0.1, 'content_type': 4.6000000000000005, 'scheme': 1.3}. Best is trial 2 with value: 705168025.0.


Best trial: 2. Best value: 7.05168e+08:   6%|▌         | 3/50 [22:22<5:50:25, 447.35s/it]

[W 2026-11-23 04:39:38,135] Trial 3 failed with parameters: {'method': 3.3000000000000003, 'host': 1.5, 'query_part': 2.6, 'path_part': 2.7, 'body_part': 0.9, 'header': 4.9, 'user_agent': 3.9000000000000004, 'request_length': 4.7, 'status_code': 4.5, 'server_protocol': 3.0, 'server_name': 4.7, 'remote_addr': 0.4, 'accept_encoding': 0.9, 'content_type': 0.2, 'scheme': 1.6} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/markb/root-py-env/lib/python3.13/site-packages/optuna/study/_optimize.py", line 210, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_9917/2899740274.py", line 30, in objective
    score, cluster_count = evaluate_weights(trial_weights)
                           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_9917/2899740274.py", line 10, in evaluate_weights
    data = cur.execute(query, (
           ~~~~~~~~~~~^^^^^^^^^
        START_DATE, END_DATE,
        ^^^^^^^^^^^^^^^^^^^^^
    ...<2

KeyboardInterrupt: 